# 66. 交互折线图（px.line）

<!-- module-learning-arc:start -->
> **Plotly 模块主线｜第 3 / 18 步：交互探索趋势、类别和变量关系**
>
> **持续应用背景：** 准备周度经营预警会：让读者通过悬停、缩放、下钻和层级探索，沿着异常、定位、行动的路径完成追问。
>
> **承接上一阶段：** 图表结构与Hover  →  **本章任务：** 交互折线图（px.line）  →  **下一步：** 交互柱状图（px.bar）
>
> **大作业连接：** 本章练习将成为《周度经营预警会：交互诊断与行动看板》的一部分，最终需要把诊断和行动视图组织成支持经营预警决策的可分享 HTML。
<!-- module-learning-arc:end -->


## 本章场景

折线图最擅长讲「它怎么变」，但要问「某个点到底是多少、哪天见顶」，光看图形往往要靠猜。



## 本章目标

学完本章，你将能够：

- **理解**：理解「交互折线图（px.line）」的适用场景、数据结构要求，以及它想帮你读出的规律。
- **操作**：能按参数用相应绘图接口画出「交互折线图（px.line）」，并做必要的美化、注释与导出。
- **迁移**：能换一份真实经营数据，独立画出同类型的「交互折线图（px.line）」并读出其中的结论。


## 66.1 适用场景

**背景引入**：折线图最擅长讲「它怎么变」，但要问「某个点到底是多少、哪天见顶」，光看图形往往要靠猜。鼠标在图上悬停一下就能读到具体的数值，这正是交互式折线图比一张静态图片更合用的地方——趋势用于判断方向，hover 用于确认拐点。这里先拿上半年的销售额，用 px.line 画出一条随时能悬停读值的折线，感受这种「先看整体、再抠细节」的分析节奏。（可以把它想成按时间把珠子串起来：x 定每颗珠子的位置，y 定珠子高低，px.line 按 x 的顺序把相邻的点一颗颗连成线；顺序一乱，线就会来回折，所以画折线前先确认 x 是否排好序。）

时间或有序阶段上的连续变化，并需要交互检查单点。


## 66.2 数据结构

有序X列与一至多列Y；长表更适合color分组。


## 66.3 本章练习任务

运行基础图表后，完成以下任务：

1. 将 markers=True 改为 markers=False，观察数据点显示对趋势可读性的影响
2. 添加 line_shape="spline" 参数，对比折线与平滑曲线的视觉差异
3. 添加 hovertemplate 自定义悬停信息格式，说明交互提示对精确读值的作用


## 66.4 图表与参数速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 基础图表 | `px.line()`、`fig.update_layout()`、`fig.show()` | 时间或有序阶段上的连续变化，并需要交互检查单点。 | 月份字符串未显式排序 |
| 进阶变体 | `monthly.melt()`、`px.line()`、`fig.update_layout()`、`fig.show()` | 在基础图表上增加分组、注释、布局或交互 | 序列单位不同仍放同一Y轴 |
| 关键参数 | `markers` | 观测点 | 月份字符串未显式排序 |
| 关键参数 | `line_shape` | 线形 | 序列单位不同仍放同一Y轴 |
| 关键参数 | `hovermode` | 悬浮模式 | 图例名称使用内部列名 |
| 关键参数 | `range_x/range_y` | 初始范围 | 月份字符串未显式排序 |


## 66.5 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


<!-- math-foundation:chapter-66 -->
### 数学推导｜趋势图中的变化量与增长率

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜先算绝对变化。** $\Delta x_t=x_t-x_{t-1}$ 保留原单位。

**第 2 步｜再除以前一期形成相对变化。** 先写倍率 $r_t=x_t/x_{t-1}$，增长率就是 $g_t=r_t-1$。

**第 3 步｜多期增长要连乘。** 从 0 期到 $T$ 期的累计增长满足

$$
\frac{x_T}{x_0}=\prod_{t=1}^{T}(1+g_t)
$$

因此不能把多期百分比简单相加，除非变化都很小且只做近似。

**把上面的关系收束为本章计算式：**

$$
\Delta x_t=x_t-x_{t-1},\qquad g_t=\frac{x_t-x_{t-1}}{x_{t-1}}
$$

**符号解释：** $\Delta x_t$ 是绝对变化，$g_t$ 是环比增长率。

**代码对应：** 排序后使用 `diff()` 与 `pct_change()`，再把结果放进 Hover 或注释。

**使用边界：** 当上一期为 0 或时间间隔不一致时，增长率需要特殊处理。


In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1️⃣ 数据导入：手建两个小表 + 读取三个公开数据集
funnel = pd.DataFrame(
    {
        "stage": ["访问", "查看商品", "加入购物车", "提交订单", "支付成功"],
        "users": [12000, 7200, 3100, 1850, 1420],
    }
)
timeline = pd.DataFrame(
    {
        "task": ["数据准备", "探索分析", "图表制作", "报告复核"],
        "start": pd.to_datetime(
            ["2026-03-01", "2026-03-04", "2026-03-08", "2026-03-12"]
        ),
        "finish": pd.to_datetime(
            ["2026-03-04", "2026-03-09", "2026-03-13", "2026-03-15"]
        ),
        "owner": ["数据", "分析", "分析", "负责人"],
    }
)
diamonds = pd.read_csv("/datasets/diamonds.csv")
flights = pd.read_csv("/datasets/flights.csv")
gapminder = pd.read_csv("/datasets/gapminder.csv")
print(f"Diamonds {len(diamonds):,} | Flights {len(flights):,} | Gapminder {len(gapminder):,} 行")


In [ ]:
# 2️⃣ 特征工程：为各章图表构造分析所需的派生字段
orders_full = diamonds.assign(
    date=pd.Timestamp("2026-01-01"),
    category=diamonds["cut"],
    region=diamonds["clarity"],
    channel=diamonds["color"],
    order_value=diamonds["price"],
    items=diamonds["carat"],
    sales=diamonds["price"],
    month="公开样本",
)
orders = orders_full.sample(5_000, random_state=55)

monthly = (
    flights.query("year == 1960")
    .rename(columns={"passengers": "sales"})
    .copy()
)
monthly["orders"] = monthly["sales"]
monthly["profit"] = monthly["sales"].rolling(3, min_periods=1).mean()

regional = orders_full.groupby(["region", "channel"], as_index=False)[
    "sales"
].sum()

hierarchy = (
    diamonds.groupby(["cut", "color"], as_index=False)["price"]
    .sum()
    .rename(
        columns={"cut": "department", "color": "category", "price": "sales"}
    )
)

countries = gapminder.query("year == 2007").assign(
    country=lambda frame: frame["country"],
    market=lambda frame: frame["country"],
    sales=lambda frame: frame["gdpPercap"],
    growth=lambda frame: frame["lifeExp"],
)
print(f"样本：orders {len(orders):,} | monthly {len(monthly):,} 行")


## 66.6 基础图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
fig = px.line(monthly, x="month", y="sales", markers=True, title="上半年销售额趋势")
fig.update_layout(
    xaxis_title="月份", yaxis_title="销售额（万元）", hovermode="x unified"
)
fig.show()


**练一练**：上一张图用了 markers=True 把销售额的观测点标了出来。练习很简单——请把下面的折线改成平滑曲线并保留数据点：

- 将 `markers` 设为 `True`（保留观测点）
- 加上 `line_shape="spline"`（把连线改成平滑曲线）

运行后用代码自检：`fig.data[0].line.shape` 应为 `"spline"`，且图上带数据点标记。


In [ ]:
# 请在下方填写代码
import plotly.express as px

# 用 monthly 数据画销售额折线，补全两个参数


In [ ]:
import plotly.express as px

fig = px.line(
    monthly,
    x="month",
    y="sales",
    markers=True,
    line_shape="spline",
    title="上半年销售额趋势",
)
fig.update_layout(
    xaxis_title="月份", yaxis_title="销售额（万元）", hovermode="x unified"
)


## 66.7 进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
long = monthly.melt(
    id_vars="month",
    value_vars=["sales", "profit"],
    var_name="metric",
    value_name="value",
)
long["metric"] = long["metric"].map({"sales": "销售额", "profit": "利润"})
fig = px.line(
    long, x="month", y="value", color="metric", markers=True, title="销售额与利润趋势"
)
fig.update_layout(
    xaxis_title="月份",
    yaxis_title="金额（万元）",
    legend_title="指标",
    hovermode="x unified",
)
fig.show()


## 66.8 参数说明

- markers：观测点
- line_shape：线形
- hovermode：悬浮模式
- range_x/range_y：初始范围


## 66.9 结果解读

默认视图应表达总体趋势，Hover用于确认拐点和精确值。


## 66.10 本章实训：交互图与信息层次

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd
import plotly.express as px

report = pd.DataFrame(
    {
        "region": ["华东", "华南", "华北", "西南"],
        "sales": [320, 250, 280, 190],
    }
)
fig = px.bar(report, x="region", y="sales", title="地区销售额")
fig.show()


### 66.10.1 第一个结果怎么读

Plotly 的基本流程是：准备表格、映射字段、设置标题、显示图形。悬停提示只能补充信息，不能替代坐标轴和单位。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
fig = px.bar(
    report.sort_values("sales", ascending=False),
    x="region",
    y="sales",
    text="sales",
    title="按销售额排序的地区销售额",
)
fig.update_traces(textposition="outside")
fig.update_layout(yaxis_title="销售额", xaxis_title="地区")
fig.show()


### 66.10.2 第二个结果怎么读

第二个实验增加数值标签并排序。请检查：标签是否遮挡、标题是否准确、图形是否仍然能在窄屏阅读。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 66.11 错误恢复：空数据还能不能画图

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd
import plotly.express as px

report = pd.DataFrame({"region": ["华东", "华南"], "sales": [120, 150]})
if report.empty:
    print("没有可绘制的数据，请先检查筛选条件。")
else:
    fig = px.bar(report, x="region", y="sales", title="地区销售额")
    fig.update_layout(yaxis_title="销售额", xaxis_title="地区")
    fig.show()


### 66.11.1 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

筛选后先判断是否为空，再调用绘图函数。空表不是绘图库的问题，而是上游筛选口径需要检查。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 66.12 易错点提醒

- 月份字符串未显式排序
- 序列单位不同仍放同一Y轴
- 图例名称使用内部列名


## 66.13 练习与作业

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


## 66.14 独立迁移练习

在默认图可读的前提下，增加一个 hover 字段或筛选交互。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# 独立迁移练习：把「销售额」换成「订单量」，并加一条目标线对比
# 【目标】换字段 + 加参考线，练习在图上补充比较基准。
import plotly.express as px

# 起点示例(已可运行)：换 y 字段，并用 add_hline 加一条均值参考线。
fig = px.line(monthly, x="month", y="orders", markers=True, title="上半年订单量趋势")
fig.add_hline(y=monthly["orders"].mean(), line_dash="dash", line_color="#d93025")
fig.update_layout(xaxis_title="月份", yaxis_title="订单量（笔）", hovermode="x unified")
fig.show()

# ---- 反思记录：加了均值参考线，图的可读性如何提升 ----
change_note = "待填写"
expected_change = "待填写"
observed_change = "运行后填写"
print(f"改动：{change_note}")
print(f"预期：{expected_change}")
print(f"观察：{observed_change}")


In [ ]:
indexed = monthly.copy()
indexed["销售额指数"] = indexed["sales"] / indexed["sales"].iloc[0] * 100
indexed["订单量指数"] = indexed["orders"] / indexed["orders"].iloc[0] * 100
long_index = indexed.melt(
    id_vars="month",
    value_vars=["销售额指数", "订单量指数"],
    var_name="指标",
    value_name="指数",
)
fig = px.line(
    long_index, x="month", y="指数", color="指标", markers=True, title="经营指标相对增长"
)
fig.add_hline(y=100, line_dash="dash", line_color="gray")
fig.show()


## 66.15 小结

用交互折线图查看时间趋势、多序列和Hover精确值。


### 66.15.1 你已经掌握

- 判断交互折线图（px.line）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 66.15.2 关键参数

| 参数 | 作用 |
| --- | --- |
| `markers` | 观测点 |
| `line_shape` | 线形 |
| `hovermode` | 悬浮模式 |
| `range_x/range_y` | 初始范围 |


### 66.15.3 需要注意

- 月份字符串未显式排序
- 序列单位不同仍放同一Y轴
- 图例名称使用内部列名


### 66.15.4 完成检查

- [ ] 能判断什么问题适合使用交互折线图（px.line）
- [ ] 能准备符合要求的数据结构
- [ ] 能独立完成基础图表和一个进阶变体
- [ ] 能调整关键参数并解释视觉变化
- [ ] 能根据图表写出有边界的数据结论


### 66.15.5 下一步推荐

把同一图表迁移到另一份数据，先保留同样的编码，再只改变一个维度。比较迁移前后的可读性，并说明哪些结论仍然成立。
